# <span style = "color:rebeccapurple"> Preprocessing</span>

<span style="text-transform: uppercase;
        font-size: 14px;
        letter-spacing: 1px;
        font-family: 'Segoe UI', sans-serif;">
    Author
</span><br>
efrén cruz cortés
<hr style="border: none; height: 1px; background: linear-gradient(to right, transparent 0%, #ccc 10%, transparent 100%); margin-top: 10px;">

**Main Points**
- Standardization
- Normalization
- Categorical Variables

## <span style = "color:darkorchid"> Imports and datasets

In [ ]:
# :: IMPORTS ::

# Scikit-learn specifics:
from sklearn import datasets
from sklearn import preprocessing

# Helper modules
import pandas as pd
import numpy as np

## <span style = "color:darkorchid">Loading our own data

Let's load the "penguins" dataset from our "data" folder using pandas. Remember we imported pandas as pd above.

In [ ]:
penguins_directory = "data/penguins.csv"
fish_directory = "data/fish.csv"

In [ ]:
penguins_df = pd.read_csv(penguins_directory)

In [ ]:
penguins_df.head()

#### <span style = "color:red"> EXERCISE

1. Load the fish dataset from our data folder as a pandas dataframe.
2. Visualize the first few rows.
3. Make a new dataframe with only the "weight" column.
4. Drop the weight column from the original dataframe, make sure the change is permanent. Hint: `.drop()` method (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html))

In [ ]:
# Read fish dataset


In [ ]:
# Show first few rows


In [ ]:
# Make new dataframe with only weigth column


In [ ]:
# Make new dataframe without weight column. Hint: .drop() method.


## <span style = "color:darkorchid">Preprocessing: Transforming data with scikit-learn

Many machine learning algorithms are based on taking distances among data points. Distances will depend on the scale of each dimension (each measured feature). Hence, if the scales are too different from each other (say one feature is measured in the thousands, and another in decimals), we will get suboptimal, if not completely disastrous, results. There are many different ways to transform your data, which one you choose will depend on the type of data and the algorithm you are using. Here are three commonly used transformations:
<ul>
    <li>Standardization</li>
    <li>Normalization</li>
    <li>Encoding categorical features</li>
</ul>

`scikit-learn` uses the module `preprocessing` to perform the operations above.

We have done the necessary imports at the top of this notebook. Now let's review these transformations one by one.

### <span style="color:teal">Standardization</span>

Standardization is a statistics based approach to bring all features to a similar scale. It computes the mean and standard deviation over observations of a feature, and then subtracts the mean from each observation and divides by the standard deviation. This results in mean $0$, standard deviation $1$ statistics.

The name *standardization* comes from computing the standard score, or z-score, of your observations. This is done in the following manner:
$$
z_i = \frac{x_i - \hat{\mu}}{\hat{\sigma}}
$$

Let's try it with one of the penguin columns.

In [ ]:
# This is how flipper length looks
penguins_df[["flipper_length_mm"]].head(10)

Did you notice I used double brackets? `scikit-learn` prefers pandas dataframes as input, and not pandas series, so even though we'll use only one column, we'll keep it as a dataframe. Similarly, if your input is a numpy array, it must be $2$-dimensional (like a matrix, even if it is a one-column matrix).

In [ ]:
# Create a standard scaler object:
z_scaler = preprocessing.StandardScaler()
z_scaler

Depending on your IDE, you may get a handy-dandy visual representation of your object. If you hover over the information icon, you will see it says *Not fitted*.

STOP!!

We just created an object, and instance of the `StandardScaler` class. The orange box above is a visual representation of such object. We'll explore now what methods and attributes it contains.

In [ ]:
# First, we can "fit" it to the data
z_scaler.fit(penguins_df[["flipper_length_mm"]])

If you get the box visualization, notice the color change!

What does fitting do? Well, the fitted scaler now has $\hat{\mu}$ and $\hat{\sigma}$. That is, the sample mean and standard deviation.

In [ ]:
print(f"Mean: {z_scaler.mean_}\nVariance: {z_scaler.var_}")

We can use the `transform` method to transform our data:

In [ ]:
# Transform the data into z-scores
z_flipper_length = z_scaler.transform(penguins_df[["flipper_length_mm"]])

In [ ]:
# Let's check our transformed data
print(f"Type: {type(z_flipper_length)}\nShape: {z_flipper_length.shape}")
z_flipper_length[0:10]

Notice that the output is a numpy array. When working with `sklearn` I recommend copious use of `numpy`, but if you are a hardcore Pandas fan, it's possible, we'll look into the method `.set_output()` later.

Is our data really mean $0$ and standard deviation $1$? Let's double check:

In [ ]:
print(f"Mean: {z_flipper_length.mean():.3f}\nStandard Deviation: {z_flipper_length.std():.6f}")

<b>Review, with two columns</b>

In [ ]:
# mock data, let's use two columns this time
x = penguins_df[["flipper_length_mm", "bill_length_mm"]]

# Create a standard scaler object
z_scaler = preprocessing.StandardScaler()

# Fit it to the data
z_scaler.fit(x)

# Transform the data
z = z_scaler.transform(x)

# Let's see the first few rows
z[0:10]

#### <span style="color:blue">Example - Building Intuition</span>

Imagine we'll perform an ML algorithm (clustering, for example), and find the situation in which our data comes in completely different scales:

In [ ]:
var1 = 'flipper_length_mm'
var2 = 'body_mass_g'
penguins_df[[var1, var2]].head()

![penguin-vitrubius](images/vitrubian_penguin.png){width=25%}

In [ ]:
# Pick sample points to track in our visualization
sample_idx = [1, 50, 100]
sample_colors = ['red', 'blue', 'green']
sample_points = [np.array([penguins_df[var1].iloc[idx], penguins_df[var2].iloc[idx]]) for idx in sample_idx]

# Standardize data
z_scaler = preprocessing.StandardScaler()
z_scaler.fit(penguins_df[[var1, var2]])
z = z_scaler.transform(penguins_df[[var1, var2]])

Let's visualize the data, highlighting three sample points.

(The code generating the image below can be found in the `support_materials.ipynb` file).

![standardization-comparison](images/standardization_comparison.png){width=90%}

**Question**

Which point is closest to RED?

In [ ]:
# If we take distances in original scale:
d_raw_rb = np.linalg.norm(sample_points[0] - sample_points[1]) # dist between red and blue point
d_raw_rg = np.linalg.norm(sample_points[0] - sample_points[2]) # dist between red and green point
print(f"red -> blue (raw): {d_raw_rb:.1f}\nred -> green (raw): {d_raw_rg:.1f}\n")

In [ ]:
# If we take distances in standard scale:
z_sample_points = [np.array([z[idx, 0], z[idx, 1]]) for idx in sample_idx]
d_std_rb = np.linalg.norm(z_sample_points[0] - z_sample_points[1])
d_std_rg = np.linalg.norm(z_sample_points[0] - z_sample_points[2])
print(f"red -> blue (standardized): {d_std_rb:.1f}\nred -> green (standardized): {d_std_rg:.1f}\n")

#### <span style = "color:red"> EXERCISE

Pick any numerical column(s) from your fish dataframe (except weight) and standardize it/them!

In [ ]:
# I'll reload the data for you
fish_df = pd.read_csv(fish_directory)

In [ ]:
# Create a standard scaler object


# Fit it to the data


# Transform the data


In [ ]:
# View the first few elements


### <span style="color:teal">Normalization</span>

While standardization is statistics based, normalization is geometry based. That means there is an important assumption: the data point to be normalized is assumed to be a vector in a vector space. If you don't know what this is, let's not worry about it at this point. For now, this means we cannot use it in categorical data.

Another important point is that normalization happens over the whole ambient space of a data point. That is, it normalizes per row. Notice that, in contrast, standardization was done with column statistics of the whole sample.

If you are interested, normalization is the process of transforming a vector so it has unit norm:
$$
x_{\text{norm}} = \frac{x}{\left||x|\right|}
$$

There are some options to pick when normalizing (for example, what "norm" to use). If you want to learn more about these, check the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.Normalizer.html)

Let's try it with the first few rows from diabetes:

In [ ]:
diabetes = datasets.load_diabetes(as_frame=True)

In [ ]:
# Rows before normalization
diabetes.data.head()

In [ ]:
# Create normalizer
norm_scaler = preprocessing.Normalizer(norm = "l2")    # <-- "l2" indicates Euclidean norm
norm_scaler

In [ ]:
# Fit to data
norm_scaler.fit(diabetes.data)

In [ ]:
# Transform data
norm_diab = norm_scaler.transform(diabetes.data)
norm_diab

And double check the rows are actually norm $1$:

In [ ]:
print(np.linalg.norm(norm_diab[0]))
print(np.linalg.norm(norm_diab[42]))


**Note**

Normalization actually does not require fitting (you are not computing a sample mean and standard deviation as in standardization). Hence, the fit method above is kind of redundant (this is why when you first created your normalizer, it was already a blue box, and not an orange one!). However, it is kept for consistency among other transformation methods.

There is actually a shortcut function that can be used directly:

In [ ]:
# Normalization using one function:
norm_diab = preprocessing.normalize(diabetes.data[0:3], norm = "l2")

norm_diab

<b>Summary</b>

In [ ]:
# data to normalize
x = diabetes.data

# Create normalizer
norm_scaler = preprocessing.Normalizer(norm = "l2")

# "Fit" to data
norm_scaler.fit(x)    # <-- doesn't really do much, but keeps syntax/logic consistent

# transform data
x_norm = norm_scaler.transform(x)

In [ ]:
# Alternatively, you can use the function shortcut:
x_norm = preprocessing.normalize(x, norm = "l2")

The advantage of creating a Normalizer object is that it can be added to a Pipeline object (we will talk about these soon).

#### <span style = "color:red"> EXERCISE

Normalize the predictive, numerical rows of the fish dataset.

In [ ]:
# I'll load the data for you, and drop the columns we don't want
fish_df = pd.read_csv(fish_directory)
fish_df = fish_df.drop(columns = ["species", "weight"])

In [ ]:
# Create normalizer


# "Fit" to data


# transform data


In [ ]:
# Visualize the first 10 elements


In [ ]:
# Check the norm of the first row is 1
np.linalg.norm() # <-- Your first row goes inside these parentheses.

### <span style="color:teal">Encoding Categorical variables

Categorical variables don't always have the nice properties of numbers (order, distance, etc.). Therefore, we have to encode them in a way we can deal with them mathematically. There are two main ways of doing this, which depend on the categories having an ordered structure or not.

#### Categorical variables with order

When your categorical variables have an order structure, you will do *ordinal* encoding, which means you will map them to the integers. For example, if you have a list of grades: $A$, $B$, $C$, etc. You know the following facts:
- $A > B$
- $A > C$
- $B > C$

Note that these facts are encoded in the integer numbers ${0, 1, 2, 3, ...}$. So we can map each letter to a number.

To exemplify this, let's make a mock dataframe:

In [ ]:
grades_df = pd.DataFrame({"Grades":["A", "A", "D", "B", "C", "A", "C"]})
grades_df

In [ ]:
# Step 1: Create the OrdinalEncoder
ord_encoder = preprocessing.OrdinalEncoder()
ord_encoder

In [ ]:
# Step 2: Fit it to our data
ord_encoder.fit(grades_df)

What did fitting do? It let the encoder know the ordered categories that exist in out data:

In [ ]:
ord_encoder.categories_

In [ ]:
# Step 3: Transform your data
ord_encoder.transform(grades_df)

You can also transform new data:

In [ ]:
new_grades_df = pd.DataFrame({"Grades":["D", "D", "B"]})

In [ ]:
# Transforming new data
ord_encoder.transform(new_grades_df)

What happens if we try to encode a previously unobserved category?

In [ ]:
# You should get an error
unseen_grades_df = pd.DataFrame({"Grades":["F", "F", "A"]})
ord_encoder.transform(unseen_grades_df)

#### Categorical variables with no order:

If your variables don't have any order whatsoever, the recommended approach is one-hot encoding. This basically maps each value to a vector whose $i^{th}$ element is $1$ if it belongs to category $i$, and $0$ otherwise.

For example, if we have efrén's favorite animals: "Aardvark", "Babirusa", and "Capybara", for which there is no natural order, the encoding could go like this:
- "Aardvark" $\rightarrow [1,0,0]$
- "Babirusa" $\rightarrow [0,1,0]$
- "Capybara" $\rightarrow [0,0,1]$

![aardvark-and-friends](images/aardvark_babirusa_capybara.png?v1){width = 50%}


Let's see this in action with the penguins island feature:

In [ ]:
# This is how the original data looks
penguins_df[["island"]]

In [ ]:
# Step 1: Create a OneHotEncoder
oh_encoder = preprocessing.OneHotEncoder()
oh_encoder

In [ ]:
# Step 2: Fit it to the data
oh_encoder.fit(penguins_df[["island"]])

In [ ]:
# Step 3: Transform the data
oh_islands = oh_encoder.transform(penguins_df[["island"]])

Note: since the result will be a matrix with a lot of zeros, scikit-learn actually returns in "compressed sparse row" format. Like his:

In [ ]:
oh_islands

We don't have time to go over this in detail, but, you have two options:
1. You can specify you don't want a sparse output with `sparse_output` argument, or
2. you can easily "decompress" it by calling the `toarray()` method:

In [ ]:
# create OneHotEncoder with sparse_output as False:
preprocessing.OneHotEncoder(sparse_output = False).fit(penguins_df[["island"]]).transform(penguins_df[["island"]])[0:10]

In [ ]:
# convert to dense array
oh_islands_arr = oh_islands.toarray()
oh_islands_arr[0:10]

How do we know which vector element belongs to which category? Use the `categories_` attribute, the order they appear on will be the order of the data:

In [ ]:
oh_encoder.categories_

Finally, you may want to get back the categories from one-hot encoded data, in that case, just use the `inverse_transform()` method.

In [ ]:
oh_raw_data = [[1,0,0], [1,0,0], [0,0,1]]
oh_encoder.inverse_transform(oh_raw_data)

#### <span style = "color:red"> EXERCISE

The first predictive feature of the fish dataset (the species) is categorical.
1. Create a one hot encoder and fit it to this data.
2. Transform the data.
3. Check the first few rows of the transformed data.

In [ ]:
# I'll load the data for you, and select the column we want
fish_df = pd.read_csv(fish_directory)
fish_df = fish_df[["species"]]

In [ ]:
# Make new encoder with updated sparse_output argument:


# Changing the output format of encoder:


# Fit it to the data


# Transform the data


In [ ]:
# Check the transformed data


### <span style="color:teal">How to choose the right scaler?

There are a variety of factors to consider. Here are a few.

**Is the data numerical or categorical?**
- If the data is numerical, then use standardization or normalization (there are other transformations we won't go over).
- If the data is categorical, then choose ordinal encoding if the categories have a natural order, or one-hot encoding if they don't.

**How is the data distributed and what are your model assumptions?**
- If the data is approximately gaussian, or if the algorithm you are using assumes gaussianity, use standardization. Example of algorithms that prefer gaussian data: linear and logistic regression, k-nearest neighbors, principal component analysis.

**Do you care about magnitude or about direction?**
- Some machine learning methods work better when vector *direction* is more relevant than magnitude. For example, most large language models today use vector representations of words in such a way that their alignment is more important than their absolute distance. In those cases, use L2 normalization.
- Many neural network algorithms and some uses of support vector machines (text classification, for example) are better suited for normalization.
- We did not go over it, but if you want to normalize and keep your results sparse (few relevant dimensions), use L1 normalization.

**Do we want to keep the data within a given range?**
- If so, you'll have to use a MinMaxScaler, which we didn't go over. It scales over a dimension/feature, like the StandardScaler, but makes sure that observations are within a range, for example in the interval $[0,1]$.

### <span style="color:teal">Summary

We learn a few classes from the `preprocessing` module:
- `StandardScaler`
- `Normalizer`
- `OrdinalEncoder`
- `OneHotEncoder`

There are many more you will learn in your scikit-learn adventures, but we can't deal with those here. Remember the main recipe for preprocessors:
1. Create and instance of the object you need. Usually like this: `preprocessor = SomeClass()`</li>
2. Fit the preprocessor, usually like this: `preprocessor.fit(X)`</li>
3. Transform your data, which may or may not be the same as the one used for fitting. Usually like this: `preprocessor.transform(X_2)`

Keep this recipe in mind, because the models we are about to use follow a similar logic!

## Extra material

**Reverse alphabetical order**

Notice that the order was alphabetical: $A$ got mapped to $0$. However, you may want the opposite: for $D$ to be $0$.

To my knowledge, there is no nice option to encode in reverse alphabetical order. However, we can provide the categories to encode as an explicit list (a list of lists to be precise, where the $i^{th}$ list corresponds to the $i^{th}$ column in your dataframe). In this case, the order in which we provide these categories will indicate the order of encoding:

In [ ]:
# Indicating explicitly the categories:
ord_encoder = preprocessing.OrdinalEncoder(categories = [["D", "C", "B", "A"]])

ord_encoder.fit(grades_df)

ord_encoder.transform(grades_df)

Now $A$ maps to the higher number.

**Data back into DF**
How do we put the data back into our dataframe?

In [ ]:
penguins_df.head()

First, we could just create a new dataframe from the array:

In [ ]:
oh_islands_df = pd.DataFrame(oh_islands_arr, columns = oh_encoder.categories_)
oh_islands_df.head()

Now we just join the dataframes:

In [ ]:
penguins_df.join(oh_islands_df).head()

Alternatively, and presumably easier, we can change the encoder's output format to a pandas dataframe using the `.set_output()` method, BUT, if that is the case, we must set `sparse_output` as `False`.

In [ ]:
# Make new encoder with updated sparse_output argument:
oh_encoder = preprocessing.OneHotEncoder(sparse_output = False)

# Changing the output format of encoder:
oh_encoder.set_output(transform = "pandas")

oh_encoder.fit(penguins_df[["island"]])

oh_islands_df = oh_encoder.transform(penguins_df[["island"]])

In [ ]:
oh_islands_df.head()

In [ ]:
penguins_df.join(oh_islands_df).head()